In [42]:
import os
import re
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

In [45]:
def read_file(file_path):
    with open(file_path, 'r') as file:
        return file.read()

def extract_operators(text):
    operators = re.findall(r'\[(AND|OR|NOT)\]', text)
    return operators

def count_operators(operators):
    return {
        'AND': operators.count('AND'),
        'OR': operators.count('OR'),
        'NOT': operators.count('NOT')
    }

def compare_operators(label_text, model_text):
    label_operators = extract_operators(label_text)
    model_operators = extract_operators(model_text)

    label_counts = count_operators(label_operators)
    model_counts = count_operators(model_operators)

    relaxed_comparison = {
        'AND': label_counts['AND'] == model_counts['AND'],
        'OR': label_counts['OR'] == model_counts['OR'],
        'NOT': label_counts['NOT'] == model_counts['NOT']
    }

    def exact_position_comparison(label_text, model_text, operator):
        pattern = re.compile(rf'(\w+)\s+\[{operator}\]')
        label_positions = [(match.group(1), match.start()) for match in pattern.finditer(label_text)]
        model_positions = [(match.group(1), match.start()) for match in pattern.finditer(model_text)]
        return label_positions == model_positions

    exact_comparison = {
        'AND': exact_position_comparison(label_text, model_text, 'AND'),
        'OR': exact_position_comparison(label_text, model_text, 'OR'),
        'NOT': exact_position_comparison(label_text, model_text, 'NOT')
    }

    return relaxed_comparison, exact_comparison, label_counts, model_counts

def calculate_metrics(true_operators, pred_operators):
    true_labels = [1 if op in true_operators else 0 for op in ['AND', 'OR', 'NOT']]
    pred_labels = [1 if op in pred_operators else 0 for op in ['AND', 'OR', 'NOT']]

    precision = precision_score(true_labels, pred_labels, average='binary')
    recall = recall_score(true_labels, pred_labels, average='binary')
    f1 = f1_score(true_labels, pred_labels, average='binary')
    accuracy = accuracy_score(true_labels, pred_labels)

    return precision, recall, f1, accuracy

def extract_nct_number(filename):
    match = re.search(r'NCT\d+', filename)
    return match.group() if match else None

def process_files(label_folder, model_folder):
    relaxed_comparisons = []
    exact_comparisons = []
    label_operator_counts = []
    model_operator_counts = []

    for model_file in os.listdir(model_folder):
        if model_file.endswith('.txt'):
            nct_number = extract_nct_number(model_file)

            if nct_number:
                label_file_path = os.path.join(label_folder, f'{nct_number}.txt')
                model_file_path = os.path.join(model_folder, model_file)

                if os.path.exists(label_file_path):
                    label_text = read_file(label_file_path)
                    model_text = read_file(model_file_path)

                    relaxed_comparison, exact_comparison, label_counts, model_counts = compare_operators(label_text, model_text)

                    relaxed_comparisons.append(relaxed_comparison)
                    exact_comparisons.append(exact_comparison)
                    label_operator_counts.append(label_counts)
                    model_operator_counts.append(model_counts)

    return relaxed_comparisons, exact_comparisons, label_operator_counts, model_operator_counts

def aggregate_metrics(label_operator_counts, model_operator_counts):
    true_operators = []
    pred_operators = []

    for label_count, model_count in zip(label_operator_counts, model_operator_counts):
        for op in ['AND', 'OR', 'NOT']:
            true_operators.extend([op] * label_count[op])
            pred_operators.extend([op] * model_count[op])

    precision, recall, f1, accuracy = calculate_metrics(true_operators, pred_operators)

    return precision, recall, f1, accuracy


In [46]:
# Pfade zu den Ordnern
label_folder = '../../input/lct_p1'
model_folder = 'model_output/Llama-3-70B-Instruct_4_shot/output'



relaxed_comparisons, exact_comparisons, label_operator_counts, model_operator_counts = process_files(label_folder, model_folder)
precision, recall, f1, accuracy = aggregate_metrics(label_operator_counts, model_operator_counts)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Accuracy:", accuracy)

In [49]:
total_and = sum(count['AND'] for count in label_operator_counts)
total_or = sum(count['OR'] for count in label_operator_counts)
total_not = sum(count['NOT'] for count in label_operator_counts)

print("Total AND:", total_and)
print("Total OR:", total_or)
print("Total NOT:", total_not)

In [50]:
total_and = sum(count['AND'] for count in model_operator_counts)
total_or = sum(count['OR'] for count in model_operator_counts)
total_not = sum(count['NOT'] for count in model_operator_counts)

print("Total AND:", total_and)
print("Total OR:", total_or)
print("Total NOT:", total_not)